# Formalia

Please read the [assignment overview page](https://laura.alessandretti.com/comsocsci2026/wiki_pages/Assignments.html) carefully before proceeding. The page contains information about formatting (including formats etc), group sizes, and many other aspects of handing in the assignment. 

__If you fail to follow these simple instructions, it will negatively impact your grade!__

**Due date and time**: The assignment is due on Mar 3rd at 23:59. Hand in your Jupyter notebook file (with extension `.ipynb`) via DTU Learn _(Assignment 1)_. 

Remember to include in the first cell of your notebook:
* the link to your group's Git repository 
* group members' contributions


## Part 1: Ready Made vs Custom Made Data

> **Exercise: Ready made data vs Custom made data** In this exercise, I want to make sure you have understood they key points of my lecture and the reading. 
>
> 1. What are pros and cons of the custom-made data used in Centola's experiment (the first study presented in the lecture) and the ready-made data used in Nicolaides's study (the second study presented in the lecture)? You can support your arguments based on the content of the lecture and the information you read in Chapter 2.3 of the book __(answer in max 150 words)__.
> 2. How do you think these differences can influence the interpretation of the results in each study? __(answer in max 150 words)__

1. Custom-made experimental data, such as that in the Centola study, is useful because it allows for proper experimental randomization, which makes causal analysis unambiguous, allowing you to be more certain that it's effects are real. However because the situation is artifical there is no way of establishing that the behavior in experimental conditions is the same as what happens in the real world. 
To overcome this you need to real-world observation data. However, such data, requires establishing using some more complex analytical tools as well as at least one convincingly exogenous shock (such as the weather used in the Nicolaides article), which is rarely easily available.
2. Because of these defects giving a straightforward interpretation of either study is difficult. To establish whether the experimental study succeeds in saying anything about spread of behavior in real-world social networks, you need to investigate in what ways the experimental setup differs from the part of the real world you want information about, and whether these differences affect the observed effects.
To interpret a study like the one by Nicolaides you need to be convinced that the claimed exogenous effect really is exogenous (in this case it's the weather, which convincingly exogenous), and study the analytical methods.

## Part 2: Find Researchers using the OpenAlex API

> **Exercise 3: Find potential Computational Social Scientists** In this exercise, we'll use the OpenAlex API to compile a list of researchers in the field of Computational Social Science, focusing on those who have attended the IC2S2 conference in 2025. This will not only later on help you understand the landscape of Computational Social Science research but also develop practical skills in data collection and analysis. 
>
> Please read the text of the whole exercise before starting to work on it. 
>
> **Steps**
> 
> 1. **Retreive data.** Consider the set of unique researcher names that you collected in Week 1, Exercise 3. Use the _authors_ endpoint of the [OpenAlex API](https://docs.openalex.org/api-entities/authors) to _search_ these researchers in the database based on their names. Loop through the list and, for each researcher in your list, find: 
>     - their _id_: The OpenAlex ID for this author.
>     - their _display\_name_: The name of the author as a single string.
>     - their _works\_api\_url_: A URL that will get you a list of all this author's works.
>     - their _h\_index_ : The h-index for this author.
>     - their _works\_count_: The number of  Works this this author has created.
>     - their _country\_code_: The country code of their last known institution
> 2. **Data Storage** Store this information in a Pandas DataFrame and save it to file.
>
>    
> **Handling Challenges**
> 
> I expect that, while working on the steps above, you will encounter several obstacles. As you complete this exercise, you are expected to:     
>
>    - Identify problems that arise.      
>    - Improve your solutions to address such problems, making reasonable decisions when data is incomplete or ambiguous.       
>
> **Reflection Questions**
>  Answer the following questions __(max 150 words for each question)__: 
>
>    - Which challenges did you encounter? How did you address them?
>    - Choose one problem you faced while collecting the data and describe your solution. Why did you choose this approach, and what impact might it have on your data? 
>      


In [179]:
import requests
import pandas as pd 
import pyalex
import pickle
import math
import itertools
import tqdm
import joblib


In [162]:
organizers = ['Yue Zheng', 'Jacob Habinek', 'Martin Arvidsson', 'Andreas Jungherr', 'Erika Fille T. Legara', 'Johannes Fagerlind', 'Yukun Jiao', 'Xiaojing Jie', 'Juhi Kulshrestha', 'Angélica Sousa da Mata', 'Marc Keuschnigg', 'Marc Sparhuber', 'Kazuki Sakamoto', 'Peter Hedström', 'Kwabena Afriyie Owusu', 'Petter Holme', 'David Garcia', 'Lauren Fahey', 'Anastasia Menshikova', 'Yize Zhang', 'Laura Lotero', 'Elliot Ash', 'Sonia Yeh', 'Hendrik Erz', 'Vsevolod Suschevskiy', 'Thomas Haase', 'Denise Pumain', 'Zhao Feng', 'Yuan Liao', 'Josef Ginnerskov', 'José Bener', 'Yun-Tsz Tsai (Clara)', 'Mirta Galesic', 'Yang Yue', 'Helpis Saleh', 'Noé Vidal-Naquet', 'May Lim', 'Nina Tahmasebi']
with open("poster_authors.pkl", "rb") as f:
    presenters = pickle.loads(f.read())
researchers = list(set(organizers).union(set(presenters)))
len(researchers)

948

In [165]:
with open("api_key", "r") as f:
    pyalex.config.api_key = str(f.read())

alex_info = []

for researcher_name in tqdm.tqdm(researchers):
    alex_info.append(pyalex.Authors().search(researcher_name).get())

100%|██████████| 948/948 [14:18<00:00,  1.10it/s]


In [166]:
def argmax(l):
    max_i = 0
    max_e = -math.inf
    for i, e in enumerate(l):
        if e > max_e:
            max_e = e
            max_i = i
    return max_i
        
def disambig(i):
    return alex_info[i][argmax([result["relevance_score"] for result in alex_info[i]])]

disambig_alex_info = [disambig(i) for i in range(len(alex_info)) if len(alex_info[i]) > 0]

In [168]:
def path_access(o, path):
    if len(path) == 0:
        return o
    else:
        next = o[path[0]]
        if next == None:
            return None
        return path_access(next, path[1:])

d = {}
for attr in [["id"], ["display_name"], ["works_api_url"], ["summary_stats", "h_index"], ["works_count"], ["last_known_institutions", 0, "country_code"]]:
    l = []
    for researcher_alex_info in disambig_alex_info:
        l.append(path_access(researcher_alex_info, attr))
    d[".".join((str(component) for component in attr))] = l

df = pd.DataFrame.from_dict(d)
df 

,id,display_name,works_api_url,summary_stats.h_index,works_count,last_known_institutions.0.country_code
0,https://openalex.org/A5017981514,Kohei Takeda,https://api.openalex.org/works?filter=author.i...,23,169,SG
1,https://openalex.org/A5051891666,Florian Keusch,https://api.openalex.org/works?filter=author.i...,25,122,DE
2,https://openalex.org/A5032880777,Michaël Maes,https://api.openalex.org/works?filter=author.i...,141,1385,US
3,https://openalex.org/A5053982410,Fábio Sartori,https://api.openalex.org/works?filter=author.i...,14,129,IT
4,https://openalex.org/A5084087666,Wanting Wang,https://api.openalex.org/works?filter=author.i...,22,152,CN
...,...,...,...,...,...,...
901,https://openalex.org/A5022729086,Bijin Joseph,https://api.openalex.org/works?filter=author.i...,2,5,IN
902,https://openalex.org/A5026595284,Xiaoyuan Yi,https://api.openalex.org/works?filter=author.i...,4,14,CN
903,https://openalex.org/A5012128678,Max R. P. Grossmann,https://api.openalex.org/works?filter=author.i...,2,9,DE
904,https://openalex.org/A5057457548,Gang Sun,https://api.openalex.org/works?filter=author.i...,81,546,CN


In [169]:
df.to_pickle("researchers.pickle")

We found that when searching for names in the openalex database, there were multiple results for each name most of the time, meaning we had to disambiguate between similarly named researchers. 
Initially we attempted to map the concepts and topics noted for each author identification candidate to the quantitative social science field in general or to the subjects of the poster. 
However, the classification of concepts in the OpenAlex database was not precise enough, and most authors could not be disambiguated in this way. 
We had to settle for choosing the author candidate with the highest relevance score from the OpenAlex search tool, which does increase the chances that we are actually analyzing data about the wrong researchers.

## Part 3: Collect Research Articles

> **Exercise 1: Collecting Research Articles from IC2S2 Authors**
>
>In this exercise, we'll leverage the OpenAlex API to gather information on research articles authored by participants of the IC2S2 2025 conference, referred to as *IC2S2 authors*. **Before you start, please ensure you read through the entire exercise.**
>
> 
> **Steps:**
>  
> 1. **Retrieve Data:** Start with the dataset of *IC2S2 authors* you collected in Week 2, Exercise 3 (called dataset D1 in the figure above). Use the OpenAlex API [works endpoint](https://docs.openalex.org/api-entities/works) to fetch their research articles. For each article, retrieve the following details:
>    - _id_: The unique OpenAlex ID for the work.
>    - _publication_year_: The year the work was published.
>    - _cited_by_count_: The number of times the work has been cited by other works.
>    - _author_ids_: The OpenAlex IDs for the authors of the work.
>    - _title_: The title of the work.
>    - _abstract_inverted_index_: The abstract of the work, formatted as an inverted index.
> 
>     **Important Note on Paging:** By default, the OpenAlex API limits responses to 25 works per request. For more efficient data retrieval, I suggest to adjust this limit to 200 works per request. Even with this adjustment, you will need to implement pagination to access all available works for a given query. This ensures you can systematically retrieve the complete set of works beyond the initial 200. Find guidance on implementing pagination [here](https://docs.openalex.org/how-to-use-the-api/get-lists-of-entities/paging#cursor-paging).
>
> 2. **Data Storage:** Organize the retrieved information into two Pandas DataFrames and save them to two files in a suitable format:
>    - Dataset D2: The *IC2S2 papers* dataset should include: *id, publication\_year, cited\_by\_count, author\_ids*.
>    - Dataset D3: The *IC2S2 abstracts* dataset should include: *id, title, abstract\_inverted\_index*.
>  
>
> **Filters:**
> To ensure the data we collect is relevant and manageable, apply the following filters:
>     
>    - Only include *IC2S2 authors* with a total work count between 5 and 5,000.    
>    - Retrieve only works that have received more than 10 citations.    
>    - Limit to works authored by fewer than 10 individuals.    
>    - Include only works relevant to Computational Social Science (focusing on: Sociology OR Psychology OR Economics OR Political Science) AND intersecting with a quantitative discipline (Mathematics OR Physics OR Computer Science), as defined by their [Concepts](https://docs.openalex.org/api-entities/works/work-object#concepts). *Note*: here we only consider Concepts at *level=0* (the most coarse definition of concepts).     
>
> **Efficiency Tips:**
> Writing efficient code in this exercise is **crucial**. To speed up your process:
> 
> - **Apply filters directly in your request:** When possible, use the [filter parameter](https://docs.openalex.org/api-entities/works/filter-works) of the *works* endpoint to apply the filters above directly in your API request, ensuring only relevant data is returned. Learn about combining multiple filters [here](https://docs.openalex.org/how-to-use-the-api/get-lists-of-entities/filter-entity-lists).  
> - **Bulk requests:** Instead of sending one request for each author, you can use the [filter parameter](https://docs.openalex.org/api-entities/works/filter-works) to query works by multiple authors in a single request. *Note: My testing suggests that can only include up to 25 authors per request.*
> - **Use multiprocessing:** Implement multiprocessing to handle multiple requests simultaneously. I highly recommmend [Joblib’s Parallel](https://joblib.readthedocs.io/en/stable/) function for that, and [tqdm](https://tqdm.github.io/) can help monitor progress of your jobs. Remember to stay within [the rate limit](https://docs.openalex.org/how-to-use-the-api/rate-limits-and-authentication) of 100 requests per second.
>
>
>   
> For reference, employing these strategies allowed me to fetch the data in about 30 seconds using 5 cores on my laptop. I obtained a dataset of approximately 25 MB (including both the *IC2S2 abstracts* and *IC2S2 papers* files).
> 
>
> **Data Overview and Reflection questions:** Answer the following questions __(max 150 words for each question)__: 
> 
> - **Dataset summary.** How many works are listed in your Dataset D2 (*IC2S2 papers*) dataframe? How many unique researchers have co-authored these works?     
> - **Efficiency in code.** Describe the strategies you implemented to make your code more efficient. How did your approach affect your code's execution time?    
> - **Filtering Criteria and Dataset Relevance** Reflect on the rationale behind setting specific thresholds for the total number of works by an author, the citation count, the number of authors per work, and the relevance of works to specific fields. How do these filtering criteria contribute to the relevance of the dataset you compiled? Do you believe any aspects of Computational Social Science research might be underrepresented or overrepresented as a result of these choices?    

The IC2S2 papers dataset contains 27337 different works authored by 42368 different authors.

To make the code more efficient we implemented all the suggestions made in the problem description. 
In particular we
1. Used api-side filtering for the citation count, author count and concept.
2. Made requests for works from 25 authors per request.
3. Increased the pagination from 25 to 200 work received per response.
4. Ran 8 of the 25-author requests in parallel. 

Which put us at half at minute for the full download.

All the filters serve the purpose of narrowing the work couunt down the most likely papers to be relavant to the conference. The citation count filter low-impact papers, as does the paper count per author minimum limit. An author that authors more than 5000 papers can't be considered relevant to each ones, so we don't want to follow that link. The concepts filter attempt to remove the works from authors in other subjects.

Subfields with a higher threshold for publishing a paper will be underrepresented as a result of the minimum paper per author filter.
In addition we previously saw that the coarse-grained concepts in OpenAlex might be insufficient to identify works within a field. In particular we note that many of the works in the resulting dataset aren't directly relavant.

In [170]:
social_concepts = ["Sociology", "Psychology", "Economics", '"Political science"'] # pyablex doesn't automatically quote
quant_concepts = ["Mathematics", "Physics", '"Computer science"']
def get_concepts_id(concept_names):
    return [pyalex.Concepts().filter(display_name=name, level=0).get()[0]["id"] for name in concept_names]

social_concept_ids = get_concepts_id(social_concepts)
quant_concept_ids = get_concepts_id(quant_concepts)


/tmp/ipykernel_14920/1165536202.py:4: DeprecationWarning: Concepts is deprecated by OpenAlex and replaced by topics.
  return [pyalex.Concepts().filter(display_name=name, level=0).get()[0]["id"] for name in concept_names]


In [181]:
def select_author_id(work):
    return {
        "id": work["id"],
        "publication_year": work["publication_year"],
        "cited_by_count": work["cited_by_count"],
        "author_ids": [authorship["author"]["id"] for authorship in work["authorships"]],
        "title": work["title"],
        "abstract_inverted_index": work["abstract_inverted_index"]
    } 

def fetch_author_list(author_info_l):
    ids = [author_info["id"] for author_info in author_info_l]
    fields = ["id", "publication_year", "cited_by_count", "authorships", "title", "abstract_inverted_index"]
    works = pyalex.Works().filter(authorships={"author": {"id": "|".join(ids)}}, authors_count="<10", cited_by_count=">10", concepts={"id": ["|".join(social_concept_ids), "|".join(quant_concept_ids)]}).select(fields).paginate(per_page=200)
    return [select_author_id(work) for work in itertools.chain(*works)]


In [197]:
from joblib import delayed
relavant_authors = [author for author in disambig_alex_info
                    if author["works_count"] > 5 and author["works_count"] < 5000]

block_size = 25
r = joblib.Parallel(n_jobs=8)(delayed(lambda i: fetch_author_list(relavant_authors[i*block_size:(i+1)*block_size]))(i) for i in range(math.ceil(len(relavant_authors)/block_size)))

In [ ]:
all_works = [work for job in r for work in job]
all_works_unique = list({work["id"]: work for work in all_works}.values())

len(all_works_unique)

25851

In [213]:
all_unique_authors_ids = list({author_id for work in all_works_unique
                         for author_id in work["author_ids"]})
len(all_unique_authors_ids)

42368

In [211]:
d = {}
for attr in ["id", "publication_year", "cited_by_count", "author_ids"]:
    l = []
    for work in all_works:
        l.append(work[attr])
    d[attr] = l

df1 = pd.DataFrame.from_dict(d)
df1.to_pickle("works.pickle")
df1

,id,publication_year,cited_by_count,author_ids
0,https://openalex.org/W2108598243,2009.0,60261,"[https://openalex.org/A5101542158, https://ope..."
1,https://openalex.org/W4239072543,2009.0,2917,"[https://openalex.org/A5101480290, https://ope..."
2,https://openalex.org/W2250384498,2016.0,1500,"[https://openalex.org/A5015683164, https://ope..."
3,https://openalex.org/W2016674662,2009.0,1407,"[https://openalex.org/A5042890616, https://ope..."
4,https://openalex.org/W1497385253,2004.0,1300,"[https://openalex.org/A5103339501, https://ope..."
...,...,...,...,...
27332,https://openalex.org/W2892182042,2019.0,12,"[https://openalex.org/A5055710645, https://ope..."
27333,https://openalex.org/W431568915,2015.0,12,"[https://openalex.org/A5086827245, https://ope..."
27334,https://openalex.org/W1542860446,2000.0,11,"[https://openalex.org/A5091405758, https://ope..."
27335,https://openalex.org/W2950292336,2016.0,11,"[https://openalex.org/A5101577090, https://ope..."


In [212]:
d = {}
for attr in ["id", "title", "abstract_inverted_index"]:
    l = []
    for work in all_works:
        l.append(work[attr])
    d[attr] = l

df2 = pd.DataFrame.from_dict(d)
df2.to_pickle("works.pickle")
df2

,id,title,abstract_inverted_index
0,https://openalex.org/W2108598243,ImageNet: A large-scale hierarchical image dat...,"{'The': [0], 'explosion': [1], 'of': [2, 56, 6..."
1,https://openalex.org/W4239072543,ImageNet: A large-scale hierarchical image dat...,"{'The': [0], 'explosion': [1], 'of': [2, 56, 6..."
2,https://openalex.org/W2250384498,YFCC100M,"{'This': [0], 'publicly': [1], 'available': [2..."
3,https://openalex.org/W2016674662,Multiscale mobility networks and the spatial s...,"{'Among': [0], 'the': [1, 8, 22, 29, 37, 49, 6..."
4,https://openalex.org/W1497385253,Activity Recognition in the Home Using Simple ...,None
...,...,...,...
27332,https://openalex.org/W2892182042,Simplicity Creates Inequity: Implications for ...,"{'Algorithms': [0], 'are': [1, 49, 219], 'incr..."
27333,https://openalex.org/W431568915,Dissecting the Spirit of Gezi: Influence vs. S...,None
27334,https://openalex.org/W1542860446,Algorithms for Constructing Comparative Maps,None
27335,https://openalex.org/W2950292336,Image-embodied Knowledge Representation Learning,"{'Entity': [0], 'images': [1, 61], 'could': [2..."
